# Importing

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.utils.class_weight import compute_sample_weight
import sys
from sklearn.model_selection import ParameterSampler

In [2]:
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [3]:
from model_evaluation import (
    operating_point_at_recall_floor,
    summarize_candidate_results,
)

# Load the frozen V3-E1 development data

In [4]:
PROJECT_DIR = Path("..").resolve()

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_DEV_DIR = RAW_DIR / "dev"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
SRC_DIR = PROJECT_DIR / "src"


In [5]:
DEV_PATH = PROCESSED_DIR / "v3e1_dev_sep1_7.parquet"

dev_df = pd.read_parquet(DEV_PATH)

print("Shape:", dev_df.shape)
print("Columns:", len(dev_df.columns))

Shape: (3731672, 30)
Columns: 30


## Configuration we are targetting

In [6]:
RANDOM_STATE = 42
N_CANDIDATES = 20


In [7]:
V4_T1_SEARCH_SPACE = {
    "model__num_leaves": [
        15,
        31,
        63,
    ],
    "model__max_depth": [
        6,
        8,
        10,
        -1,
    ],
    "model__min_child_samples": [
        50,
        100,
        250,
        500,
    ],
    "model__learning_rate": [
        0.03,
        0.05,
        0.08,
    ],
    "model__n_estimators": [
        200,
        300,
        500,
    ],
    "model__reg_alpha": [
        0.0,
        0.1,
        0.5,
        1.0,
    ],
    "model__reg_lambda": [
        0.0,
        0.5,
        1.0,
        2.0,
        5.0,
    ],
    "model__subsample": [
        0.8,
        0.9,
        1.0,
    ],
    "model__colsample_bytree": [
        0.75,
        0.90,
        1.0,
    ],
}


In [8]:
candidate_params = list(
    ParameterSampler(
        param_distributions=V4_T1_SEARCH_SPACE,
        n_iter=N_CANDIDATES,
        random_state=RANDOM_STATE,
    )
)

In [9]:
candidate_table = pd.DataFrame(candidate_params)

In [10]:
candidate_table

,model__subsample,model__reg_lambda,model__reg_alpha,model__num_leaves,model__n_estimators,model__min_child_samples,model__max_depth,model__learning_rate,model__colsample_bytree
0,0.8,0.0,0.1,63,200,100,-1,0.05,0.75
1,1.0,0.5,0.1,63,300,100,6,0.03,0.75
2,1.0,0.5,0.1,63,200,250,-1,0.08,1.00
3,0.9,0.0,1.0,63,300,100,8,0.03,1.00
4,0.9,2.0,0.1,63,300,500,10,0.03,0.75
5,0.8,2.0,1.0,31,500,50,8,0.05,0.90
6,0.9,0.0,0.5,15,500,100,6,0.08,0.90
7,1.0,1.0,0.1,63,300,500,-1,0.03,1.00
8,0.8,0.5,0.0,15,500,100,-1,0.05,0.75
9,1.0,0.5,1.0,15,200,50,-1,0.05,0.90


In [11]:
candidate_table.insert(
    0,
    "candidate_id",
    [f"V4-T1-{i:02d}" for i in range(1, len(candidate_table) + 1)],
)
candidate_table

,candidate_id,model__subsample,model__reg_lambda,model__reg_alpha,model__num_leaves,model__n_estimators,model__min_child_samples,model__max_depth,model__learning_rate,model__colsample_bytree
0,V4-T1-01,0.8,0.0,0.1,63,200,100,-1,0.05,0.75
1,V4-T1-02,1.0,0.5,0.1,63,300,100,6,0.03,0.75
2,V4-T1-03,1.0,0.5,0.1,63,200,250,-1,0.08,1.00
3,V4-T1-04,0.9,0.0,1.0,63,300,100,8,0.03,1.00
4,V4-T1-05,0.9,2.0,0.1,63,300,500,10,0.03,0.75
5,V4-T1-06,0.8,2.0,1.0,31,500,50,8,0.05,0.90
6,V4-T1-07,0.9,0.0,0.5,15,500,100,6,0.08,0.90
7,V4-T1-08,1.0,1.0,0.1,63,300,500,-1,0.03,1.00
8,V4-T1-09,0.8,0.5,0.0,15,500,100,-1,0.05,0.75
9,V4-T1-10,1.0,0.5,1.0,15,200,50,-1,0.05,0.90


## a folder for checkpoints

In [12]:
from pathlib import Path

RESULTS_DIR = Path("tuning_results")
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [13]:
candidate_table.to_csv(
    RESULTS_DIR / "v4_t1_candidate_parameters.csv",
    index=False,
)

# Freeze the V3-E1 feature set

In [14]:
V3E1_NUMERIC_FEATURES = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
    "sender_tx_count_1h",
    "log_sender_amount_paid_sum_24h_same_currency",
    "log_sender_hours_since_prev_tx",
    "log_amount_paid_vs_sender_median",
    "receiver_seen_before",
    "sender_receiver_prior_tx_count",
    "sender_distinct_receivers_24h",
]

V3E1_CATEGORICAL_FEATURES = [
    "receiving_currency",
    "payment_currency",
    "payment_format",
]

FEATURE_COLUMNS_V3E1 = V3E1_NUMERIC_FEATURES + V3E1_CATEGORICAL_FEATURES

TARGET_COLUMN = "is_laundering"
TIMESTAMP_COLUMN = "transaction_timestamp"

## Checks

In [15]:
required_columns = FEATURE_COLUMNS_V3E1 + [TARGET_COLUMN, TIMESTAMP_COLUMN]

missing_columns = [col for col in required_columns if col not in dev_df.columns]

if missing_columns:
    raise ValueError(f"Missing required V3-E1 columns: {missing_columns}")


dev_df[TIMESTAMP_COLUMN] = pd.to_datetime(dev_df[TIMESTAMP_COLUMN])

if not dev_df[TIMESTAMP_COLUMN].is_monotonic_increasing:
    raise ValueError(
        "Development data is not chronologically sorted. "
        "Do not sort it automatically yet because we need "
        "to preserve the exact frozen fold membership."
    )


EXPECTED_DEV_ROWS = 3_731_672

if len(dev_df) != EXPECTED_DEV_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_DEV_ROWS:,} development rows, but found {len(dev_df):,}."
    )


if dev_df[TIMESTAMP_COLUMN].max() >= pd.Timestamp("2022-09-08"):
    raise ValueError(
        "September 8+ data is present. This tuning notebook must contain only Sept 1–7."
    )


print("V3-E1 feature count:", len(FEATURE_COLUMNS_V3E1))
print("Rows:", f"{len(dev_df):,}")
print(
    "Time range:",
    dev_df[TIMESTAMP_COLUMN].min(),
    "to",
    dev_df[TIMESTAMP_COLUMN].max(),
)
print(
    "Fraud prevalence:",
    f"{dev_df[TARGET_COLUMN].mean():.6%}",
)

V3-E1 feature count: 17
Rows: 3,731,672
Time range: 2022-09-01 00:00:00 to 2022-09-07 23:59:00
Fraud prevalence: 0.081116%


# Create X and y

In [16]:
X_dev = dev_df[FEATURE_COLUMNS_V3E1]
y_dev = dev_df[TARGET_COLUMN].astype(int)

print("X shape:", X_dev.shape)
print("y shape:", y_dev.shape)

X shape: (3731672, 17)
y shape: (3731672,)


In [17]:
frozen_folds = [
    (
        slice(0, 2_076_752),
        slice(2_076_752, 2_284_182),
    ),
    (
        slice(0, 2_284_182),
        slice(2_284_182, 2_766_832),
    ),
    (
        slice(0, 2_766_832),
        slice(2_766_832, 3_731_672),
    ),
]

In [18]:
for fold_number, (train_slice, val_slice) in enumerate(
    frozen_folds,
    start=1,
):
    X_train = X_dev.iloc[train_slice]
    X_val = X_dev.iloc[val_slice]

    print(f"Fold {fold_number}: Train={len(X_train):,} | Validation={len(X_val):,}")

Fold 1: Train=2,076,752 | Validation=207,430
Fold 2: Train=2,284,182 | Validation=482,650
Fold 3: Train=2,766,832 | Validation=964,840


# Rebuild the frozen preprocessing

In [19]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

In [20]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
        ),
    ]
)

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            V3E1_NUMERIC_FEATURES,
        ),
        (
            "categorical",
            categorical_pipeline,
            V3E1_CATEGORICAL_FEATURES,
        ),
    ]
)

# Rebuild the untuned V3-E1 LightGBM

In [22]:
benchmark_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=8,
    min_child_samples=100,
    reg_alpha=0.0,
    reg_lambda=1.0,
    subsample=1.0,
    colsample_bytree=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

In [23]:
v3e1_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            benchmark_model,
        ),
    ]
)

# Evaluator 

In [24]:
def evaluate_lightgbm_candidate(
    candidate_id,
    candidate_params,
    pipeline_template,
    X,
    y,
    frozen_folds,
    recall_floor=0.80,
):
    """
    Evaluate one LightGBM hyperparameter configuration
    across all frozen chronological folds.
    """

    fold_results = []

    for fold_number, (
        train_slice,
        val_slice,
    ) in enumerate(
        frozen_folds,
        start=1,
    ):
        print(f"\n{candidate_id} — Fold {fold_number}")

        # ---------------------------------------------
        # 1. Get this fold's chronological data
        # ---------------------------------------------

        X_train = X.iloc[train_slice]
        y_train = y.iloc[train_slice]

        X_val = X.iloc[val_slice]
        y_val = y.iloc[val_slice]

        print(f"Train: {len(X_train):,} | Validation: {len(X_val):,}")

        # ---------------------------------------------
        # 2. Calculate imbalance weights using ONLY
        #    this fold's training labels
        # ---------------------------------------------

        train_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_train,
        )

        # ---------------------------------------------
        # 3. Create a completely fresh pipeline
        # ---------------------------------------------

        pipeline = clone(pipeline_template)

        # Apply this candidate's hyperparameters.
        pipeline.set_params(**candidate_params)

        # ---------------------------------------------
        # 4. Fit
        # ---------------------------------------------

        start_time = time.perf_counter()

        pipeline.fit(
            X_train,
            y_train,
            model__sample_weight=train_weights,
        )

        fit_seconds = time.perf_counter() - start_time

        # ---------------------------------------------
        # 5. Score future validation transactions
        # ---------------------------------------------

        y_score = pipeline.predict_proba(X_val)[:, 1]

        # ---------------------------------------------
        # 6. Primary ranking metric
        # ---------------------------------------------

        average_precision = average_precision_score(
            y_val,
            y_score,
        )

        # ---------------------------------------------
        # 7. Secondary metrics
        # ---------------------------------------------

        roc_auc = roc_auc_score(
            y_val,
            y_score,
        )

        prevalence = y_val.mean()

        pr_lift = average_precision / prevalence

        # ---------------------------------------------
        # 8. Operational behavior at >=80% recall
        # ---------------------------------------------

        operational = operating_point_at_recall_floor(
            y_true=y_val,
            y_score=y_score,
            min_recall=recall_floor,
        )

        # ---------------------------------------------
        # 9. Record results
        # ---------------------------------------------

        result = {
            "candidate_id": candidate_id,
            "fold": fold_number,
            "average_precision": average_precision,
            "pr_lift": pr_lift,
            "roc_auc": roc_auc,
            "fit_seconds": fit_seconds,
            **operational,
        }

        fold_results.append(result)

        print(f"AP: {average_precision:.6f}")

        print(
            f"Precision @ >=80% recall: {operational['precision_at_recall_floor']:.4%}"
        )

        print(f"Recall: {operational['recall_at_recall_floor']:.4%}")

        print(f"False positives: {operational['false_positives_at_recall_floor']:,}")

        print(f"Alert rate: {operational['alert_rate_at_recall_floor']:.4%}")

        print(f"Fit time: {fit_seconds:.2f}s")

    return pd.DataFrame(fold_results)

# First run: reproduce V3-E1

In [25]:
V3E1_BENCHMARK_PARAMS = {
    "model__n_estimators": 300,
    "model__learning_rate": 0.05,
    "model__num_leaves": 31,
    "model__max_depth": 8,
    "model__min_child_samples": 100,
    "model__reg_alpha": 0.0,
    "model__reg_lambda": 1.0,
    "model__subsample": 1.0,
    "model__colsample_bytree": 1.0,
}

In [26]:
benchmark_check = evaluate_lightgbm_candidate(
    candidate_id="V3-E1-BENCHMARK",
    candidate_params=V3E1_BENCHMARK_PARAMS,
    pipeline_template=v3e1_pipeline,
    X=X_dev,
    y=y_dev,
    frozen_folds=frozen_folds,
    recall_floor=0.80,
)

benchmark_check


V3-E1-BENCHMARK — Fold 1
Train: 2,076,752 | Validation: 207,430


AP: 0.377397
Precision @ >=80% recall: 6.5044%
Recall: 80.0983%
False positives: 4,686
Alert rate: 2.4162%
Fit time: 37.45s

V3-E1-BENCHMARK — Fold 2
Train: 2,284,182 | Validation: 482,650
AP: 0.302327
Precision @ >=80% recall: 1.5986%
Recall: 80.0425%
False positives: 23,206
Alert rate: 4.8861%
Fit time: 39.34s

V3-E1-BENCHMARK — Fold 3
Train: 2,766,832 | Validation: 964,840
AP: 0.402554
Precision @ >=80% recall: 6.0333%
Recall: 80.0584%
False positives: 12,818
Alert rate: 1.4138%
Fit time: 53.24s


,candidate_id,fold,average_precision,pr_lift,roc_auc,fit_seconds,diagnostic_cutoff,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,true_positives_at_recall_floor,false_negatives_at_recall_floor,true_negatives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,V3-E1-BENCHMARK,1,0.377397,192.342899,0.983886,37.445404,0.851777,0.065044,0.800983,4686,326,81,202337,5012,0.024162
1,V3-E1-BENCHMARK,2,0.302327,309.804799,0.976807,39.343648,0.338196,0.015986,0.800425,23206,377,94,458973,23583,0.048861
2,V3-E1-BENCHMARK,3,0.402554,377.820990,0.983962,53.238797,0.718563,0.060333,0.800584,12818,823,205,950994,13641,0.014138


In [27]:
benchmark_check[
    [
        "fold",
        "average_precision",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
        "fit_seconds",
    ]
]

,fold,average_precision,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor,fit_seconds
0,1,0.377397,0.065044,0.800983,4686,5012,0.024162,37.445404
1,2,0.302327,0.015986,0.800425,23206,23583,0.048861,39.343648
2,3,0.402554,0.060333,0.800584,12818,13641,0.014138,53.238797


# Run only Candidate 1

In [28]:
candidate_1_params = candidate_params[0]

candidate_1_params

{'model__subsample': 0.8,
 'model__reg_lambda': 0.0,
 'model__reg_alpha': 0.1,
 'model__num_leaves': 63,
 'model__n_estimators': 200,
 'model__min_child_samples': 100,
 'model__max_depth': -1,
 'model__learning_rate': 0.05,
 'model__colsample_bytree': 0.75}

In [29]:
v3e1_pipeline.set_params(model__subsample_freq=1)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, defau

In [30]:
candidate_1_results = evaluate_lightgbm_candidate(
    candidate_id="V4-T1-01",
    candidate_params=candidate_1_params,
    pipeline_template=v3e1_pipeline,
    X=X_dev,
    y=y_dev,
    frozen_folds=frozen_folds,
    recall_floor=0.80,
)


V4-T1-01 — Fold 1
Train: 2,076,752 | Validation: 207,430


AP: 0.391576
Precision @ >=80% recall: 6.6982%
Recall: 80.0983%
False positives: 4,541
Alert rate: 2.3463%
Fit time: 36.55s

V4-T1-01 — Fold 2
Train: 2,284,182 | Validation: 482,650
AP: 0.299530
Precision @ >=80% recall: 1.9032%
Recall: 80.0425%
False positives: 19,432
Alert rate: 4.1042%
Fit time: 39.06s

V4-T1-01 — Fold 3
Train: 2,766,832 | Validation: 964,840
AP: 0.417984
Precision @ >=80% recall: 5.0478%
Recall: 80.0584%
False positives: 15,481
Alert rate: 1.6898%
Fit time: 47.94s


In [31]:
candidate_1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
        "fit_seconds",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor,fit_seconds
0,1,0.391576,199.568822,0.984640,0.066982,0.800983,4541,4867,0.023463,36.547838
1,2,0.299530,306.938562,0.977891,0.019032,0.800425,19432,19809,0.041042,39.062552
2,3,0.417984,392.302965,0.985030,0.050478,0.800584,15481,16304,0.016898,47.940725


In [32]:
candidate_1_summary = {
    "candidate_id": "V4-T1-01",
    "mean_ap": candidate_1_results["average_precision"].mean(),
    "std_ap": candidate_1_results["average_precision"].std(),
    "min_ap": candidate_1_results["average_precision"].min(),
    "max_ap": candidate_1_results["average_precision"].max(),
    "ap_range": (
        candidate_1_results["average_precision"].max()
        - candidate_1_results["average_precision"].min()
    ),
    "mean_precision_at_80_recall": candidate_1_results[
        "precision_at_recall_floor"
    ].mean(),
    "mean_false_positives": candidate_1_results[
        "false_positives_at_recall_floor"
    ].mean(),
    "mean_alert_rate": candidate_1_results["alert_rate_at_recall_floor"].mean(),
    "total_fit_seconds": candidate_1_results["fit_seconds"].sum(),
}

pd.Series(candidate_1_summary)

candidate_id                       V4-T1-01
mean_ap                            0.369696
std_ap                             0.062184
min_ap                              0.29953
max_ap                             0.417984
ap_range                           0.118454
mean_precision_at_80_recall        0.045497
mean_false_positives           13151.333333
mean_alert_rate                    0.027135
total_fit_seconds                123.551115
dtype: object

In [33]:
candidate_1_summary = summarize_candidate_results(
    candidate_id="V4-T1-01",
    fold_results=candidate_1_results,
    candidate_params=candidate_params[0],
)

pd.Series(candidate_1_summary)

candidate_id                       V4-T1-01
mean_ap                            0.369696
std_ap                             0.062184
min_ap                              0.29953
max_ap                             0.417984
ap_range                           0.118454
fold1_ap                           0.391576
fold2_ap                            0.29953
fold3_ap                           0.417984
mean_roc_auc                       0.982521
mean_precision_at_80_recall        0.045497
mean_false_positives           13151.333333
mean_alert_rate                    0.027135
total_fit_seconds                123.551115
subsample                               0.8
reg_lambda                              0.0
reg_alpha                               0.1
num_leaves                               63
n_estimators                            200
min_child_samples                       100
max_depth                                -1
learning_rate                          0.05
colsample_bytree                

In [34]:
all_fold_results = candidate_1_results.copy()

all_candidate_summaries = [candidate_1_summary]

# Run Candidates 2 through 20

In [35]:
import gc


for candidate_number in range(
    2,
    len(candidate_params) + 1,
):
    candidate_id = f"V4-T1-{candidate_number:02d}"

    params = candidate_params[candidate_number - 1]

    print("\n" + "=" * 70)
    print(f"Evaluating {candidate_id} ({candidate_number}/{len(candidate_params)})")
    print("=" * 70)

    # -----------------------------------------
    # Evaluate this configuration on all
    # three chronological folds
    # -----------------------------------------

    candidate_results = evaluate_lightgbm_candidate(
        candidate_id=candidate_id,
        candidate_params=params,
        pipeline_template=v3e1_pipeline,
        X=X_dev,
        y=y_dev,
        frozen_folds=frozen_folds,
        recall_floor=0.80,
    )

    # -----------------------------------------
    # Save fold-level results
    # -----------------------------------------

    all_fold_results = pd.concat(
        [
            all_fold_results,
            candidate_results,
        ],
        ignore_index=True,
    )

    # -----------------------------------------
    # Create candidate-level summary
    # -----------------------------------------

    candidate_summary = summarize_candidate_results(
        candidate_id=candidate_id,
        fold_results=candidate_results,
        candidate_params=params,
    )

    all_candidate_summaries.append(candidate_summary)

    # -----------------------------------------
    # Checkpoint after EVERY candidate
    # -----------------------------------------

    all_fold_results.to_csv(
        RESULTS_DIR / "v4_t1_fold_results.csv",
        index=False,
    )

    pd.DataFrame(all_candidate_summaries).to_csv(
        RESULTS_DIR / "v4_t1_candidate_summary.csv",
        index=False,
    )

    # Help release objects between runs.
    del candidate_results
    gc.collect()


Evaluating V4-T1-02 (2/20)

V4-T1-02 — Fold 1
Train: 2,076,752 | Validation: 207,430


AP: 0.348168
Precision @ >=80% recall: 5.0115%
Recall: 80.0983%
False positives: 6,179
Alert rate: 3.1360%
Fit time: 38.26s

V4-T1-02 — Fold 2
Train: 2,284,182 | Validation: 482,650
AP: 0.259328
Precision @ >=80% recall: 1.7694%
Recall: 80.0425%
False positives: 20,930
Alert rate: 4.4146%
Fit time: 43.42s

V4-T1-02 — Fold 3
Train: 2,766,832 | Validation: 964,840
AP: 0.335672
Precision @ >=80% recall: 5.1067%
Recall: 80.0584%
False positives: 15,293
Alert rate: 1.6703%
Fit time: 51.75s

Evaluating V4-T1-03 (3/20)

V4-T1-03 — Fold 1
Train: 2,076,752 | Validation: 207,430
AP: 0.404564
Precision @ >=80% recall: 6.2452%
Recall: 80.0983%
False positives: 4,894
Alert rate: 2.5165%
Fit time: 36.02s

V4-T1-03 — Fold 2
Train: 2,284,182 | Validation: 482,650
AP: 0.342166
Precision @ >=80% recall: 1.3627%
Recall: 80.0425%
False positives: 27,288
Alert rate: 5.7319%
Fit time: 43.02s

V4-T1-03 — Fold 3
Train: 2,766,832 | Validation: 964,840
AP: 0.439840
Precision @ >=80% recall: 4.3051%
Recall: 80.0

In [36]:
candidate_summary_df = pd.DataFrame(all_candidate_summaries)

candidate_summary_df.shape

(20, 23)

In [37]:
candidate_ranking = candidate_summary_df.sort_values(
    "mean_ap",
    ascending=False,
).reset_index(drop=True)

candidate_ranking[
    [
        "candidate_id",
        "mean_ap",
        "std_ap",
        "ap_range",
        "fold1_ap",
        "fold2_ap",
        "fold3_ap",
        "mean_precision_at_80_recall",
        "mean_false_positives",
        "mean_alert_rate",
        "total_fit_seconds",
    ]
].head(10)

,candidate_id,mean_ap,std_ap,ap_range,fold1_ap,fold2_ap,fold3_ap,mean_precision_at_80_recall,mean_false_positives,mean_alert_rate,total_fit_seconds
0,V4-T1-03,0.395523,0.049461,0.097675,0.404564,0.342166,0.439840,0.039710,16825.333333,0.034099,131.495841
1,V4-T1-16,0.390713,0.051529,0.095403,0.413216,0.331760,0.427163,0.038753,16677.333333,0.032817,103.630563
2,V4-T1-14,0.386372,0.062320,0.120087,0.405642,0.316693,0.436780,0.042291,15075.000000,0.030882,149.474398
3,V4-T1-05,0.385280,0.052351,0.095940,0.409489,0.325205,0.421145,0.046227,11795.666667,0.025263,143.287182
4,V4-T1-17,0.379487,0.038968,0.077657,0.383288,0.338758,0.416414,0.048188,11361.000000,0.023720,103.943834
5,V4-T1-08,0.378606,0.054548,0.104186,0.397292,0.317170,0.421356,0.039803,14958.333333,0.030932,150.874221
6,V4-T1-01,0.369696,0.062184,0.118454,0.391576,0.299530,0.417984,0.045497,13151.333333,0.027135,123.551115
7,V4-T1-06,0.367479,0.032287,0.061883,0.356830,0.341862,0.403745,0.052991,10621.000000,0.022503,184.463476
8,V4-T1-11,0.366732,0.043797,0.083465,0.382075,0.317328,0.400793,0.048396,11631.666667,0.025138,142.902323
9,V4-T1-04,0.361917,0.070817,0.127084,0.407409,0.280324,0.398019,0.044868,12278.000000,0.026140,148.571862


In [38]:
PARAMETER_COLUMNS = [
    "num_leaves",
    "max_depth",
    "min_child_samples",
    "learning_rate",
    "n_estimators",
    "reg_alpha",
    "reg_lambda",
    "subsample",
    "colsample_bytree",
]

comparison_columns = [
    "candidate_id",
    "mean_ap",
    "std_ap",
    "fold1_ap",
    "fold2_ap",
    "fold3_ap",
    "mean_precision_at_80_recall",
    "mean_false_positives",
    "mean_alert_rate",
] + PARAMETER_COLUMNS


candidate_ranking[comparison_columns].head(10)

,candidate_id,mean_ap,std_ap,fold1_ap,fold2_ap,fold3_ap,mean_precision_at_80_recall,mean_false_positives,mean_alert_rate,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,reg_alpha,reg_lambda,subsample,colsample_bytree
0,V4-T1-03,0.395523,0.049461,0.404564,0.342166,0.439840,0.039710,16825.333333,0.034099,63,-1,250,0.08,200,0.1,0.5,1.0,1.00
1,V4-T1-16,0.390713,0.051529,0.413216,0.331760,0.427163,0.038753,16677.333333,0.032817,63,10,50,0.05,200,0.0,0.5,1.0,1.00
2,V4-T1-14,0.386372,0.062320,0.405642,0.316693,0.436780,0.042291,15075.000000,0.030882,63,-1,250,0.03,300,0.5,0.5,1.0,1.00
3,V4-T1-05,0.385280,0.052351,0.409489,0.325205,0.421145,0.046227,11795.666667,0.025263,63,10,500,0.03,300,0.1,2.0,0.9,0.75
4,V4-T1-17,0.379487,0.038968,0.383288,0.338758,0.416414,0.048188,11361.000000,0.023720,31,-1,100,0.05,500,1.0,0.5,0.9,1.00
5,V4-T1-08,0.378606,0.054548,0.397292,0.317170,0.421356,0.039803,14958.333333,0.030932,63,-1,500,0.03,300,0.1,1.0,1.0,1.00
6,V4-T1-01,0.369696,0.062184,0.391576,0.299530,0.417984,0.045497,13151.333333,0.027135,63,-1,100,0.05,200,0.1,0.0,0.8,0.75
7,V4-T1-06,0.367479,0.032287,0.356830,0.341862,0.403745,0.052991,10621.000000,0.022503,31,8,50,0.05,500,1.0,2.0,0.8,0.90
8,V4-T1-11,0.366732,0.043797,0.382075,0.317328,0.400793,0.048396,11631.666667,0.025138,31,-1,50,0.05,300,0.1,1.0,0.8,1.00
9,V4-T1-04,0.361917,0.070817,0.407409,0.280324,0.398019,0.044868,12278.000000,0.026140,63,8,100,0.03,300,1.0,0.0,0.9,1.00


In [39]:
PARAMETER_COLUMNS = [
    "num_leaves",
    "max_depth",
    "min_child_samples",
    "learning_rate",
    "n_estimators",
    "reg_alpha",
    "reg_lambda",
    "subsample",
    "colsample_bytree",
]

In [40]:
parameter_effect_tables = {}

for parameter in PARAMETER_COLUMNS:
    table = (
        candidate_summary_df.groupby(parameter)
        .agg(
            candidate_count=(
                "candidate_id",
                "count",
            ),
            mean_ap=(
                "mean_ap",
                "mean",
            ),
            median_ap=(
                "mean_ap",
                "median",
            ),
            mean_ap_std=(
                "std_ap",
                "mean",
            ),
            mean_precision_80=(
                "mean_precision_at_80_recall",
                "mean",
            ),
            mean_false_positives=(
                "mean_false_positives",
                "mean",
            ),
            mean_alert_rate=(
                "mean_alert_rate",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            "mean_ap",
            ascending=False,
        )
    )

    parameter_effect_tables[parameter] = table

    print("\n" + "=" * 70)
    print(parameter.upper())
    print("=" * 70)

    display(table)


NUM_LEAVES


,num_leaves,candidate_count,mean_ap,median_ap,mean_ap_std,mean_precision_80,mean_false_positives,mean_alert_rate
2,63,9,0.370174,0.378606,0.057195,0.042421,14254.740741,0.029464
1,31,6,0.352118,0.361216,0.045449,0.048336,11515.555556,0.024621
0,15,5,0.300663,0.290839,0.042768,0.044860,12198.600000,0.026539



MAX_DEPTH


,max_depth,candidate_count,mean_ap,median_ap,mean_ap_std,mean_precision_80,mean_false_positives,mean_alert_rate
0,-1,8,0.359909,0.374151,0.048379,0.044482,13363.791667,0.027941
3,10,4,0.351199,0.370490,0.050611,0.045152,13177.750000,0.027890
2,8,4,0.349177,0.355493,0.052514,0.046748,12227.250000,0.025685
1,6,4,0.316704,0.319728,0.050441,0.043163,12462.166667,0.026944



MIN_CHILD_SAMPLES


...

,reg_lambda,candidate_count,mean_ap,median_ap,mean_ap_std,mean_precision_80,mean_false_positives,mean_alert_rate
3,2.0,2,0.376379,0.376379,0.042319,0.049609,11208.333333,0.023883
2,1.0,3,0.356802,0.366732,0.053618,0.044207,12744.333333,0.027032
0,0.0,5,0.354581,0.355700,0.059309,0.046728,12311.266667,0.025861
1,0.5,9,0.337031,0.330603,0.046195,0.042945,13728.962963,0.028946
4,5.0,1,0.318246,0.318246,0.043499,0.044121,12612.333333,0.026922



SUBSAMPLE


,subsample,candidate_count,mean_ap,median_ap,mean_ap_std,mean_precision_80,mean_false_positives,mean_alert_rate
0,0.8,5,0.356716,0.366732,0.044200,0.048216,11921.000000,0.024845
2,1.0,7,0.344422,0.378606,0.052765,0.040956,14826.857143,0.031125
1,0.9,8,0.344132,0.346111,0.051367,0.046041,11873.250000,0.025438



COLSAMPLE_BYTREE


,colsample_bytree,candidate_count,mean_ap,median_ap,mean_ap_std,mean_precision_80,mean_false_positives,mean_alert_rate
2,1.00,10,0.360659,0.372669,0.054433,0.043517,13686.100000,0.028499
0,0.75,6,0.341085,0.343152,0.046041,0.045778,12279.222222,0.026234
1,0.90,4,0.323624,0.327384,0.045179,0.046565,11960.666667,0.025803


In [41]:
top_n = 5

top_candidates = candidate_summary_df.nlargest(
    top_n,
    "mean_ap",
)

top_candidates[
    [
        "candidate_id",
        "mean_ap",
    ]
    + PARAMETER_COLUMNS
]

,candidate_id,mean_ap,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,reg_alpha,reg_lambda,subsample,colsample_bytree
2,V4-T1-03,0.395523,63,-1,250,0.08,200,0.1,0.5,1.0,1.00
15,V4-T1-16,0.390713,63,10,50,0.05,200,0.0,0.5,1.0,1.00
13,V4-T1-14,0.386372,63,-1,250,0.03,300,0.5,0.5,1.0,1.00
4,V4-T1-05,0.385280,63,10,500,0.03,300,0.1,2.0,0.9,0.75
16,V4-T1-17,0.379487,31,-1,100,0.05,500,1.0,0.5,0.9,1.00


In [42]:
for parameter in PARAMETER_COLUMNS:
    print(f"\n{parameter}")

    display(
        top_candidates[parameter]
        .value_counts()
        .rename_axis(parameter)
        .reset_index(name="top5_count")
    )


num_leaves


,num_leaves,top5_count
0,63,4
1,31,1



max_depth


,max_depth,top5_count
0,-1,3
1,10,2



min_child_samples


...

,reg_lambda,top5_count
0,0.5,4
1,2.0,1



subsample


,subsample,top5_count
0,1.0,3
1,0.9,2



colsample_bytree


,colsample_bytree,top5_count
0,1.00,4
1,0.75,1


# Stage 2

In [43]:
BOOSTING_PAIRS = [
    # Aggressive / shorter boosting
    (0.08, 200),
    # Middle ground
    (0.05, 300),
    (0.05, 500),
    # Slower / more gradual boosting
    (0.03, 300),
    (0.03, 500),
]

In [44]:
V4_T2_RANKING_SPACE = {
    # 63 won Stage 1 and was our upper boundary,
    # so test modestly beyond it.
    "num_leaves": [
        63,
        95,
        127,
    ],
    "max_depth": [
        10,
        -1,
    ],
    "min_child_samples": [
        100,
        250,
        500,
    ],
    "reg_alpha": [
        0.0,
        0.1,
        0.5,
    ],
    # 0.5 dominated top-5 AP,
    # while 2.0 performed well overall.
    "reg_lambda": [
        0.5,
        1.0,
        2.0,
    ],
    "subsample": [
        0.9,
        1.0,
    ],
    "colsample_bytree": [
        0.9,
        1.0,
    ],
}

In [45]:
V4_T2_BALANCED_SPACE = {
    "num_leaves": [
        31,
        63,
        95,
    ],
    "max_depth": [
        8,
        10,
        -1,
    ],
    "min_child_samples": [
        100,
        250,
        500,
    ],
    # Stronger regularization is allowed here.
    "reg_alpha": [
        0.5,
        1.0,
    ],
    "reg_lambda": [
        1.0,
        2.0,
    ],
    # Stage 1 suggests partial row sampling
    # often helped operational metrics.
    "subsample": [
        0.8,
        0.9,
    ],
    "colsample_bytree": [
        0.75,
        0.9,
        1.0,
    ],
}

In [46]:
import numpy as np
import pandas as pd


STAGE2_RANDOM_STATE = 42

rng = np.random.default_rng(STAGE2_RANDOM_STATE)

In [47]:
def sample_stage2_candidates(
    search_space,
    boosting_pairs,
    n_candidates,
    prefix,
    rng,
):
    """
    Randomly sample constrained LightGBM configurations.

    learning_rate and n_estimators are sampled together
    as a sensible pair rather than independently.
    """

    candidates = []

    seen = set()

    while len(candidates) < n_candidates:
        learning_rate, n_estimators = boosting_pairs[rng.integers(len(boosting_pairs))]

        params = {
            "model__num_leaves": int(rng.choice(search_space["num_leaves"])),
            "model__max_depth": int(rng.choice(search_space["max_depth"])),
            "model__min_child_samples": int(
                rng.choice(search_space["min_child_samples"])
            ),
            "model__learning_rate": (learning_rate),
            "model__n_estimators": (n_estimators),
            "model__reg_alpha": float(rng.choice(search_space["reg_alpha"])),
            "model__reg_lambda": float(rng.choice(search_space["reg_lambda"])),
            "model__subsample": float(rng.choice(search_space["subsample"])),
            "model__colsample_bytree": float(
                rng.choice(search_space["colsample_bytree"])
            ),
        }

        # Make a hashable representation so
        # we don't create duplicate candidates.
        signature = tuple(sorted(params.items()))

        if signature in seen:
            continue

        seen.add(signature)

        candidate_id = f"{prefix}-{len(candidates) + 1:02d}"

        candidates.append(
            {
                "candidate_id": candidate_id,
                "params": params,
            }
        )

    return candidates

In [48]:
ranking_candidates = sample_stage2_candidates(
    search_space=V4_T2_RANKING_SPACE,
    boosting_pairs=BOOSTING_PAIRS,
    n_candidates=8,
    prefix="V4-T2-R",
    rng=rng,
)


balanced_candidates = sample_stage2_candidates(
    search_space=V4_T2_BALANCED_SPACE,
    boosting_pairs=BOOSTING_PAIRS,
    n_candidates=8,
    prefix="V4-T2-B",
    rng=rng,
)


stage2_candidates = ranking_candidates + balanced_candidates

In [49]:
stage2_candidate_table = pd.DataFrame(
    [
        {
            "candidate_id": candidate["candidate_id"],
            **{
                key.replace(
                    "model__",
                    "",
                ): value
                for key, value in candidate["params"].items()
            },
        }
        for candidate in stage2_candidates
    ]
)


stage2_candidate_table

,candidate_id,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,reg_alpha,reg_lambda,subsample,colsample_bytree
0,V4-T2-R-01,127,-1,250,0.08,200,0.1,2.0,0.9,1.00
1,V4-T2-R-02,63,-1,500,0.05,300,0.5,2.0,1.0,1.00
2,V4-T2-R-03,63,-1,250,0.05,500,0.1,1.0,0.9,1.00
3,V4-T2-R-04,95,10,500,0.03,300,0.1,1.0,0.9,0.90
4,V4-T2-R-05,95,-1,100,0.08,200,0.5,2.0,0.9,1.00
5,V4-T2-R-06,127,-1,250,0.08,200,0.0,2.0,0.9,1.00
6,V4-T2-R-07,127,-1,100,0.03,300,0.1,1.0,0.9,0.90
7,V4-T2-R-08,63,-1,500,0.05,500,0.5,2.0,0.9,1.00
8,V4-T2-B-01,31,-1,250,0.05,500,0.5,1.0,0.9,0.75
9,V4-T2-B-02,31,-1,250,0.05,500,0.5,1.0,0.9,1.00


In [50]:
stage2_candidates = [
    # =====================================================
    # REGION A — PR-AUC / ranking-focused
    # Anchor = V4-T1-03
    # =====================================================
    {
        "candidate_id": "V4-T2-R-01",
        "params": {
            "model__num_leaves": 95,
            "model__max_depth": -1,
            "model__min_child_samples": 250,
            "model__learning_rate": 0.08,
            "model__n_estimators": 200,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 0.5,
            "model__subsample": 1.0,
            "model__colsample_bytree": 1.0,
        },
    },
    {
        "candidate_id": "V4-T2-R-02",
        "params": {
            "model__num_leaves": 127,
            "model__max_depth": -1,
            "model__min_child_samples": 250,
            "model__learning_rate": 0.08,
            "model__n_estimators": 200,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 0.5,
            "model__subsample": 1.0,
            "model__colsample_bytree": 1.0,
        },
    },
    {
        "candidate_id": "V4-T2-R-03",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": 10,
            "model__min_child_samples": 250,
            "model__learning_rate": 0.08,
            "model__n_estimators": 200,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 0.5,
            "model__subsample": 1.0,
            "model__colsample_bytree": 1.0,
        },
    },
    {
        "candidate_id": "V4-T2-R-04",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": -1,
            "model__min_child_samples": 100,
            "model__learning_rate": 0.08,
            "model__n_estimators": 200,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 0.5,
            "model__subsample": 1.0,
            "model__colsample_bytree": 1.0,
        },
    },
    {
        "candidate_id": "V4-T2-R-05",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": -1,
            "model__min_child_samples": 500,
            "model__learning_rate": 0.08,
            "model__n_estimators": 200,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 0.5,
            "model__subsample": 1.0,
            "model__colsample_bytree": 1.0,
        },
    },
    {
        "candidate_id": "V4-T2-R-06",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": -1,
            "model__min_child_samples": 250,
            "model__learning_rate": 0.05,
            "model__n_estimators": 300,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 0.5,
            "model__subsample": 1.0,
            "model__colsample_bytree": 1.0,
        },
    },
    # =====================================================
    # REGION B — balanced / operational-focused
    # Anchor = V4-T1-05
    # =====================================================
    {
        "candidate_id": "V4-T2-B-01",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": 10,
            "model__min_child_samples": 500,
            "model__learning_rate": 0.05,
            "model__n_estimators": 300,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 2.0,
            "model__subsample": 0.9,
            "model__colsample_bytree": 0.75,
        },
    },
    {
        "candidate_id": "V4-T2-B-02",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": 10,
            "model__min_child_samples": 500,
            "model__learning_rate": 0.03,
            "model__n_estimators": 500,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 2.0,
            "model__subsample": 0.9,
            "model__colsample_bytree": 0.75,
        },
    },
    {
        "candidate_id": "V4-T2-B-03",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": 10,
            "model__min_child_samples": 500,
            "model__learning_rate": 0.03,
            "model__n_estimators": 300,
            "model__reg_alpha": 0.5,
            "model__reg_lambda": 2.0,
            "model__subsample": 0.9,
            "model__colsample_bytree": 0.75,
        },
    },
    {
        "candidate_id": "V4-T2-B-04",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": 10,
            "model__min_child_samples": 500,
            "model__learning_rate": 0.03,
            "model__n_estimators": 300,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 2.0,
            "model__subsample": 0.8,
            "model__colsample_bytree": 0.75,
        },
    },
    {
        "candidate_id": "V4-T2-B-05",
        "params": {
            "model__num_leaves": 63,
            "model__max_depth": 10,
            "model__min_child_samples": 500,
            "model__learning_rate": 0.03,
            "model__n_estimators": 300,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 2.0,
            "model__subsample": 0.9,
            "model__colsample_bytree": 0.90,
        },
    },
    {
        "candidate_id": "V4-T2-B-06",
        "params": {
            "model__num_leaves": 95,
            "model__max_depth": 10,
            "model__min_child_samples": 500,
            "model__learning_rate": 0.03,
            "model__n_estimators": 300,
            "model__reg_alpha": 0.1,
            "model__reg_lambda": 2.0,
            "model__subsample": 0.9,
            "model__colsample_bytree": 0.75,
        },
    },
]

In [51]:
stage2_candidate_table = pd.DataFrame(
    [
        {
            "candidate_id": candidate["candidate_id"],
            **{
                key.replace("model__", ""): value
                for key, value in candidate["params"].items()
            },
        }
        for candidate in stage2_candidates
    ]
)

stage2_candidate_table

,candidate_id,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,reg_alpha,reg_lambda,subsample,colsample_bytree
0,V4-T2-R-01,95,-1,250,0.08,200,0.1,0.5,1.0,1.00
1,V4-T2-R-02,127,-1,250,0.08,200,0.1,0.5,1.0,1.00
2,V4-T2-R-03,63,10,250,0.08,200,0.1,0.5,1.0,1.00
3,V4-T2-R-04,63,-1,100,0.08,200,0.1,0.5,1.0,1.00
4,V4-T2-R-05,63,-1,500,0.08,200,0.1,0.5,1.0,1.00
5,V4-T2-R-06,63,-1,250,0.05,300,0.1,0.5,1.0,1.00
6,V4-T2-B-01,63,10,500,0.05,300,0.1,2.0,0.9,0.75
7,V4-T2-B-02,63,10,500,0.03,500,0.1,2.0,0.9,0.75
8,V4-T2-B-03,63,10,500,0.03,300,0.5,2.0,0.9,0.75
9,V4-T2-B-04,63,10,500,0.03,300,0.1,2.0,0.8,0.75


In [52]:
v3e1_pipeline.set_params(model__subsample_freq=1)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, defau

In [53]:
STAGE2_RESULTS_DIR = RESULTS_DIR / "stage2"

STAGE2_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [54]:
stage2_candidate_table.to_csv(
    STAGE2_RESULTS_DIR / "v4_t2_candidate_parameters.csv",
    index=False,
)

In [55]:
import gc


stage2_fold_results_list = []
stage2_candidate_summaries = []


for candidate_number, candidate in enumerate(
    stage2_candidates,
    start=1,
):
    candidate_id = candidate["candidate_id"]
    params = candidate["params"]

    print("\n" + "=" * 75)
    print(f"Evaluating {candidate_id} ({candidate_number}/{len(stage2_candidates)})")
    print("=" * 75)

    # -------------------------------------------------
    # Evaluate this candidate over all 3 frozen folds
    # -------------------------------------------------

    candidate_results = evaluate_lightgbm_candidate(
        candidate_id=candidate_id,
        candidate_params=params,
        pipeline_template=v3e1_pipeline,
        X=X_dev,
        y=y_dev,
        frozen_folds=frozen_folds,
        recall_floor=0.80,
    )

    # Keep fold-level results.
    stage2_fold_results_list.append(candidate_results)

    # -------------------------------------------------
    # Create one-row candidate summary
    # -------------------------------------------------

    candidate_summary = summarize_candidate_results(
        candidate_id=candidate_id,
        fold_results=candidate_results,
        candidate_params=params,
    )

    stage2_candidate_summaries.append(candidate_summary)

    # -------------------------------------------------
    # Checkpoint everything completed so far
    # -------------------------------------------------

    stage2_fold_results = pd.concat(
        stage2_fold_results_list,
        ignore_index=True,
    )

    stage2_summary_df = pd.DataFrame(stage2_candidate_summaries)

    stage2_fold_results.to_csv(
        STAGE2_RESULTS_DIR / "v4_t2_fold_results.csv",
        index=False,
    )

    stage2_summary_df.to_csv(
        STAGE2_RESULTS_DIR / "v4_t2_candidate_summary.csv",
        index=False,
    )

    print(f"\nCheckpoint saved after {candidate_id}")

    # Free temporary objects.
    del candidate_results
    gc.collect()


Evaluating V4-T2-R-01 (1/12)

V4-T2-R-01 — Fold 1
Train: 2,076,752 | Validation: 207,430
AP: 0.392381
Precision @ >=80% recall: 6.4072%
Recall: 80.0983%
False positives: 4,762
Alert rate: 2.4529%
Fit time: 33.33s

V4-T2-R-01 — Fold 2
Train: 2,284,182 | Validation: 482,650
AP: 0.336432
Precision @ >=80% recall: 1.6593%
Recall: 80.0425%
False positives: 22,343
Alert rate: 4.7073%
Fit time: 37.57s

V4-T2-R-01 — Fold 3
Train: 2,766,832 | Validation: 964,840
AP: 0.452437
Precision @ >=80% recall: 3.3914%
Recall: 80.0584%
False positives: 23,444
Alert rate: 2.5151%
Fit time: 41.93s

Checkpoint saved after V4-T2-R-01

Evaluating V4-T2-R-02 (2/12)

V4-T2-R-02 — Fold 1
Train: 2,076,752 | Validation: 207,430
AP: 0.416741
Precision @ >=80% recall: 6.1848%
Recall: 80.0983%
False positives: 4,945
Alert rate: 2.5411%
Fit time: 35.14s

V4-T2-R-02 — Fold 2
Train: 2,284,182 | Validation: 482,650
AP: 0.354462
Precision @ >=80% recall: 1.7660%
Recall: 80.0425%
False positives: 20,971
Alert rate: 4.4231%

In [56]:
print(
    "Candidates completed:",
    stage2_summary_df["candidate_id"].nunique(),
)

print(
    "Fold evaluations:",
    len(stage2_fold_results),
)

Candidates completed: 12
Fold evaluations: 36


In [57]:
stage2_ranking = stage2_summary_df.sort_values(
    "mean_ap",
    ascending=False,
).reset_index(drop=True)

In [58]:
stage2_ranking[
    [
        "candidate_id",
        "mean_ap",
        "std_ap",
        "ap_range",
        "fold1_ap",
        "fold2_ap",
        "fold3_ap",
        "mean_precision_at_80_recall",
        "mean_false_positives",
        "mean_alert_rate",
        "total_fit_seconds",
    ]
]

,candidate_id,mean_ap,std_ap,ap_range,fold1_ap,fold2_ap,fold3_ap,mean_precision_at_80_recall,mean_false_positives,mean_alert_rate,total_fit_seconds
0,V4-T2-R-02,0.409537,0.051849,0.102945,0.416741,0.354462,0.457407,0.036740,17297.000000,0.032472,118.061507
1,V4-T2-R-04,0.402899,0.041615,0.082870,0.407362,0.359232,0.442102,0.038180,16449.333333,0.032461,96.227808
2,V4-T2-B-02,0.400486,0.039233,0.070585,0.420275,0.355299,0.425884,0.044073,12668.666667,0.026718,140.401364
3,V4-T2-R-06,0.398296,0.045090,0.089942,0.402077,0.351434,0.441376,0.039000,15622.666667,0.031574,122.614692
4,V4-T2-B-01,0.396486,0.041164,0.078170,0.411401,0.349944,0.428114,0.046099,12157.333333,0.025698,131.716480
5,V4-T2-B-06,0.395277,0.047401,0.087853,0.415847,0.341065,0.428918,0.044940,12376.666667,0.026398,100.748140
6,V4-T2-R-01,0.393750,0.058015,0.116005,0.392381,0.336432,0.452437,0.038193,16849.666667,0.032251,112.831304
7,V4-T2-R-05,0.392003,0.044902,0.089658,0.389046,0.348653,0.438311,0.037342,16363.666667,0.031494,106.423041
8,V4-T2-B-05,0.385957,0.048501,0.094448,0.398725,0.332349,0.426797,0.046802,12587.666667,0.026797,71.477695
9,V4-T2-B-04,0.383702,0.045034,0.079897,0.407706,0.331750,0.411648,0.046106,12100.333333,0.025899,3593.895754


In [59]:
anchor_ids = [
    "V4-T1-03",
    "V4-T1-05",
]

stage1_anchors = candidate_summary_df[
    candidate_summary_df["candidate_id"].isin(anchor_ids)
].copy()

In [60]:
stage2_with_anchors = pd.concat(
    [
        stage1_anchors,
        stage2_summary_df,
    ],
    ignore_index=True,
)

In [61]:
stage2_comparison = stage2_with_anchors.sort_values(
    "mean_ap",
    ascending=False,
).reset_index(drop=True)

In [62]:
stage2_comparison[
    [
        "candidate_id",
        "mean_ap",
        "std_ap",
        "fold1_ap",
        "fold2_ap",
        "fold3_ap",
        "mean_precision_at_80_recall",
        "mean_false_positives",
        "mean_alert_rate",
        "num_leaves",
        "max_depth",
        "min_child_samples",
        "learning_rate",
        "n_estimators",
        "reg_alpha",
        "reg_lambda",
        "subsample",
        "colsample_bytree",
    ]
]

,candidate_id,mean_ap,std_ap,fold1_ap,fold2_ap,fold3_ap,mean_precision_at_80_recall,mean_false_positives,mean_alert_rate,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,reg_alpha,reg_lambda,subsample,colsample_bytree
0,V4-T2-R-02,0.409537,0.051849,0.416741,0.354462,0.457407,0.036740,17297.000000,0.032472,127,-1,250,0.08,200,0.1,0.5,1.0,1.00
1,V4-T2-R-04,0.402899,0.041615,0.407362,0.359232,0.442102,0.038180,16449.333333,0.032461,63,-1,100,0.08,200,0.1,0.5,1.0,1.00
2,V4-T2-B-02,0.400486,0.039233,0.420275,0.355299,0.425884,0.044073,12668.666667,0.026718,63,10,500,0.03,500,0.1,2.0,0.9,0.75
3,V4-T2-R-06,0.398296,0.045090,0.402077,0.351434,0.441376,0.039000,15622.666667,0.031574,63,-1,250,0.05,300,0.1,0.5,1.0,1.00
4,V4-T2-B-01,0.396486,0.041164,0.411401,0.349944,0.428114,0.046099,12157.333333,0.025698,63,10,500,0.05,300,0.1,2.0,0.9,0.75
5,V4-T1-03,0.395523,0.049461,0.404564,0.342166,0.439840,0.039710,16825.333333,0.034099,63,-1,250,0.08,200,0.1,0.5,1.0,1.00
6,V4-T2-B-06,0.395277,0.047401,0.415847,0.341065,0.428918,0.044940,12376.666667,0.026398,95,10,500,0.03,300,0.1,2.0,0.9,0.75
7,V4-T2-R-01,0.393750,0.058015,0.392381,0.336432,0.452437,0.038193,16849.666667,0.032251,95,-1,250,0.08,200,0.1,0.5,1.0,1.00
8,V4-T2-R-05,0.392003,0.044902,0.389046,0.348653,0.438311,0.037342,16363.666667,0.031494,63,-1,500,0.08,200,0.1,0.5,1.0,1.00
9,V4-T2-B-05,0.385957,0.048501,0.398725,0.332349,0.426797,0.046802,12587.666667,0.026797,63,10,500,0.03,300,0.1,2.0,0.9,0.90


# freeze

In [ ]:
WINNING_MODEL_ID = "V4-T2-B-02"

WINNING_PARAMS = {
    "model__num_leaves": 63,
    "model__max_depth": 10,
    "model__min_child_samples": 500,
    "model__learning_rate": 0.03,
    "model__n_estimators": 500,
    "model__reg_alpha": 0.1,
    "model__reg_lambda": 2.0,
    "model__subsample": 0.9,
    "model__colsample_bytree": 0.75,
    "model__subsample_freq": 1,
}

In [64]:
winner_results = stage2_summary_df.loc[
    stage2_summary_df["candidate_id"] == WINNING_MODEL_ID
].copy()

winner_results

,candidate_id,mean_ap,std_ap,min_ap,max_ap,ap_range,fold1_ap,fold2_ap,fold3_ap,mean_roc_auc,...,total_fit_seconds,num_leaves,max_depth,min_child_samples,learning_rate,n_estimators,reg_alpha,reg_lambda,subsample,colsample_bytree
7,V4-T2-B-02,0.400486,0.039233,0.355299,0.425884,0.070585,0.420275,0.355299,0.425884,0.982702,...,140.401364,63,10,500,0.03,500,0.1,2.0,0.9,0.75


In [65]:
winner_results.to_csv(
    STAGE2_RESULTS_DIR / "v4_lightgbm_winning_configuration_summary.csv",
    index=False,
)

In [66]:
import json


winner_metadata = {
    "model_id": WINNING_MODEL_ID,
    "feature_version": "V3-E1",
    "selection_stage": "V4-T2",
    "selection_reason": (
        "Strong PR-AUC improvement with improved temporal "
        "stability and lower false-positive/alert burden "
        "than the PR-AUC-maximizing candidate."
    ),
    "hyperparameters": {
        key.replace("model__", ""): value for key, value in WINNING_PARAMS.items()
    },
    "threshold_selected": False,
    "test_set_evaluated": False,
}


with open(
    STAGE2_RESULTS_DIR / "v4_lightgbm_winner_metadata.json",
    "w",
) as f:
    json.dump(
        winner_metadata,
        f,
        indent=4,
    )